## 1. Setup

In [ ]:
# Add src to Python path
import sys
sys.path.append('../src')

# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Import project utilities
from config import get_config, get_path, get_random_seed
from utils import load_csv_with_dates, calculate_metrics, set_random_seeds, save_forecast
from preprocessing import validate_monthly_data, calculate_water_balance
from features import calculate_spi, calculate_spei, identify_drought_events
from evaluation import (
    calculate_forecast_metrics,
    plot_forecast_vs_actual,
    plot_residuals,
    print_metrics_summary
)

print("✅ All imports successful!")

## 2. Configuration Management

In [ ]:
# Load configuration
config = get_config()

# Access paths
print("Data path:", get_path('processed_data'))
print("Model path:", get_path('models'))

# Access preprocessing settings
print("\nPreprocessing config:")
print("ET0 aggregation:", config.get('preprocessing.et0_aggregation'))
print("Max forward fill:", config.get('preprocessing.max_ffill_days'), "days")

# Access model settings
print("\nLSTM config:")
print("Look-back window:", config.get('forecasting.lstm.look_back'), "months")
print("Layer 1 units:", config.get('forecasting.lstm.layers')[0]['units'])

# Set random seeds for reproducibility
set_random_seeds(get_random_seed('numpy'))

## 3. Load and Validate Data

In [ ]:
# Load processed data using utility function
df = load_csv_with_dates(str(get_path('processed_data')), date_col='date')
df = df.set_index('date')

print(f"Data shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
# Validate data quality
validation_report = validate_monthly_data(df)

print("Validation Report:")
print(f"Valid: {validation_report['valid']}")

if validation_report['warnings']:
    print("\n⚠️ Warnings:")
    for warning in validation_report['warnings']:
        print(f"  - {warning}")

if validation_report['errors']:
    print("\n❌ Errors:")
    for error in validation_report['errors']:
        print(f"  - {error}")
else:
    print("\n✅ No errors found!")

## 4. Calculate Drought Indices

In [ ]:
# Calculate SPI-12 (already in v2, but showing how to recalculate)
if 'precipitation_sum' in df.columns:
    spi_12_calc, precip_rolling = calculate_spi(
        df['precipitation_sum'],
        window=12,
        distribution='gamma'
    )
    
    print(f"SPI-12 range: {spi_12_calc.min():.2f} to {spi_12_calc.max():.2f}")
    print(f"Months with SPI < -1 (drought): {(spi_12_calc < -1).sum()}")
    print(f"Months with SPI < -1.5 (severe drought): {(spi_12_calc < -1.5).sum()}")

In [ ]:
# Calculate SPEI-12
if 'precipitation_sum' in df.columns and 'et0_fao_sum' in df.columns:
    spei_12_calc, wb_rolling = calculate_spei(
        df['precipitation_sum'],
        df['et0_fao_sum'],
        window=12
    )
    
    print(f"SPEI-12 range: {spei_12_calc.min():.2f} to {spei_12_calc.max():.2f}")
    
    # Identify drought events
    drought_events = identify_drought_events(
        spei_12_calc,
        threshold=-1.0,
        min_duration=3
    )
    
    if len(drought_events) > 0:
        print(f"\nFound {len(drought_events)} drought events:")
        print(drought_events[['start_date', 'end_date', 'duration', 'severity']].head())
    else:
        print("\nNo drought events found.")

## 5. Dummy Forecast Example (Using Evaluation Utils)

In [ ]:
# Create dummy forecast for demonstration
# In real notebook, this would be your SARIMA/LSTM predictions

target = 'temperature_2m_mean'
if target in df.columns:
    # Use last 24 months as "test set"
    test_size = 24
    y_true = df[target].iloc[-test_size:].values
    dates = df.index[-test_size:]
    
    # Dummy prediction (naive: add some noise to actual)
    y_pred = y_true + np.random.normal(0, 0.5, size=len(y_true))
    
    print(f"Evaluating forecast on {test_size} months...")

In [ ]:
# Calculate metrics using utility
if target in df.columns:
    metrics = calculate_forecast_metrics(y_true, y_pred, prefix='test_')
    
    # Pretty print
    print_metrics_summary(metrics, model_name='Dummy Model')

In [ ]:
# Plot forecast vs actual
if target in df.columns:
    fig = plot_forecast_vs_actual(
        dates=dates,
        y_true=y_true,
        y_pred=y_pred,
        title='Dummy Forecast Example',
        figsize=(14, 6)
    )
    plt.show()

In [ ]:
# Plot residual diagnostics
if target in df.columns:
    fig = plot_residuals(
        y_true=y_true,
        y_pred=y_pred,
        dates=dates,
        figsize=(14, 8)
    )
    plt.show()

## 6. Save Forecast (Example)

In [ ]:
# Create forecast DataFrame
if target in df.columns:
    forecast_df = pd.DataFrame({
        'date': dates,
        'actual': y_true,
        'predicted': y_pred,
        'residual': y_true - y_pred
    })
    
    # Save using utility (creates directory if needed)
    # Uncomment to actually save:
    # output_path = save_forecast(
    #     forecast_df,
    #     filename='example_forecast.csv',
    #     output_dir=get_path('predictions')
    # )
    # print(f"Forecast saved to: {output_path}")
    
    print("Forecast DataFrame:")
    print(forecast_df.head())

## Summary

This notebook demonstrated:

✅ **Configuration management** via `config.py`  
✅ **Data loading** with `load_csv_with_dates()`  
✅ **Data validation** with `validate_monthly_data()`  
✅ **Drought analysis** with `calculate_spi()`, `calculate_spei()`  
✅ **Model evaluation** with `calculate_forecast_metrics()`  
✅ **Visualization** with `plot_forecast_vs_actual()`, `plot_residuals()`  
✅ **Result saving** with `save_forecast()`  

### Next Steps

1. **Integrate into existing notebooks**: Replace repetitive code with utility functions
2. **Customize config.yaml**: Adjust parameters for your specific needs
3. **Add more utilities**: Create helper functions for your specific workflow
4. **Document your changes**: Update IMPROVEMENTS.md with your additions

Happy forecasting! 🚀